# SCUQ-RRG Quickstart

End-to-end demo using the 5 bundled example reports. **No MIMIC-CXR access required.**

Requirements:
- GPU (for `ReportUncertaintyScorer` / GREEN model)
- `pip install -e ..` from the repo root
- `pip install -e ../third_party/GREEN/` and `pip install -e ../third_party/CXR-Report-Metric/`

In [ ]:
import pickle
import numpy as np

# Load bundled example data (5 reports, 10 samples each)
with open("prediction_5.pkl", "rb") as f:
    original_reports = pickle.load(f)   # list of 5 greedy-decoded reports

with open("samples_5.pkl", "rb") as f:
    sampled_reports = pickle.load(f)    # list of 5 x 10 stochastic samples

print(f"Loaded {len(original_reports)} reports, each with {len(sampled_reports[0])} samples.")
print("\nExample report:")
print(original_reports[0])

## 1. Build ReportSample objects

`ReportSample` is the standard input type across all SCUQ methods.

In [ ]:
from scuq import ReportSample

samples = [
    ReportSample(
        study_id=f"example_{i}",
        original_report=original_reports[i],
        sampled_reports=list(sampled_reports[i]),
    )
    for i in range(len(original_reports))
]
print(f"Created {len(samples)} ReportSample objects.")

## 2. Report-level UQ — VRO-GREEN

Measures how semantically consistent the original report is with its stochastic samples.  
`uncertainty ∈ [0, 1]`, higher = more uncertain.

In [ ]:
from scuq import ReportUncertaintyScorer

scorer = ReportUncertaintyScorer(method="vro_green", cuda=True, batch_size=10)

report_uncertainties = scorer.score_batch(samples)

for i, (u, s) in enumerate(zip(report_uncertainties, samples)):
    print(f"[{s.study_id}]  uncertainty = {u:.4f}")

## 3. Sentence-level UQ — VRO-RadGraph

Scores each sentence in the original report by measuring entity-level consistency across samples.

In [ ]:
from scuq import SentenceUncertaintyScorer

sent_scorer = SentenceUncertaintyScorer(method="vro_radgraph")
sent_results = sent_scorer.score_batch(samples)

for i, result in enumerate(sent_results):
    print(f"\n[example_{i}] per-sentence uncertainties:")
    for sent, unc in zip(result.sentences, result.uncertainty_scores):
        flag = " ← flagged" if unc == max(result.uncertainty_scores) else ""
        print(f"  {unc:.3f}  {sent[:80]}{flag}")

## 4. Calibration Evaluation

To evaluate calibration you need oracle factuality scores (e.g., GREEN scores against ground-truth reports).  
Here we simulate them for illustration — replace with real GREEN scores in a full experiment.

In [ ]:
from scuq.calibration import evaluate_calibration, plot_calibration_curve

# In a full experiment these come from src/uq/VroGreen.py output
# Using the demo outputs as a stand-in (5 samples only — illustrative)
simulated_factuality = np.array([0.40, 0.42, 0.34, 0.69, 0.50])
uncertainty_scores   = np.array(report_uncertainties)

metrics = evaluate_calibration(uncertainty_scores, simulated_factuality, n_bins=3)
print("Calibration metrics:")
for k, v in metrics.items():
    print(f"  {k:12s}: {v:.4f}")

fig = plot_calibration_curve(uncertainty_scores, simulated_factuality, method_name="VRO-GREEN")
fig.savefig("quickstart_calibration.png", dpi=150)
print("\nCalibration curve saved to quickstart_calibration.png")

## 5. DataFrame workflow

`score_dataframe()` lets you work directly with CSV files — useful for large-scale experiments.

In [ ]:
import pandas as pd
import json

df = pd.DataFrame({
    "study_id":       [f"example_{i}" for i in range(len(original_reports))],
    "original_report": original_reports,
    "sampled_reports": [json.dumps(list(s)) for s in sampled_reports],
})

df_out = scorer.score_dataframe(df)
df_out[["study_id", "uncertainty"]]

## Next steps

- **Full RaDialog experiment**: see `src/uq/VroGreen.py` and `src/uq/VroRadSent.py`
- **Abstention analysis**: `src/abstention/report_abstention.py`
- **Alignment / correlation**: `src/alignment/report_alignment.py`
- **Hallucination detection**: `src/hallucination/hallucination.py`
- **Paper results**: see `RESULTS.md`